In [2]:
%pip install dotenv


  Using cached dotenv-0.9.9-py2.py3-none-any.whl.metadata (279 bytes)
  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
Using cached dotenv-0.9.9-py2.py3-none-any.whl (1.9 kB)
Using cached python_dotenv-1.2.1-py3-none-any.whl (21 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
%pip install langchain langchain-openai


  Using cached langchain-1.1.0-py3-none-any.whl.metadata (4.9 kB)
  Using cached langchain_openai-1.1.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached openai-2.8.1-py3-none-any.whl.metadata (29 kB)
  Using cached tiktoken-0.12.0-cp312-cp312-win_amd64.whl.metadata (6.9 kB)
  Using cached langgraph_checkpoint-3.0.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached langgraph_prebuilt-1.0.5-py3-none-any.whl.metadata (5.2 kB)
  Using cached xxhash-3.6.0-cp312-cp312-win_amd64.whl.metadata (13 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached jiter-0.12.0-cp312-cp312-win_amd64.whl.metadata (5.3 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached regex-2025.11.3-cp312-cp312-win_amd64.whl.metadata (41 kB)
  Using cached ormsgpack-1.12.0-cp312-cp312-win_amd64.whl.metadata (1.2 kB)
Using cached langchain-1.1.0-py3-none-any.whl (101 kB)
Using cached langchain_openai-1.1.0-py3-none-any.whl (84 kB)
   ------------------------------------


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Literal
from typing_extensions import Annotated

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field
from IPython.display import Image, display

# Load env
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

# Initialize LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7,
    api_key=api_key
)

In [7]:
%pip install deepagents tavily-python

  Using cached wcmatch-10.1-py3-none-any.whl.metadata (5.1 kB)
  Using cached bracex-2.6-py3-none-any.whl.metadata (3.6 kB)
  Using cached docstring_parser-0.17.0-py3-none-any.whl.metadata (3.5 kB)
   ---------------------------------------- 0.0/52.0 kB ? eta -:--:--
   ---------------------------------------- 52.0/52.0 kB 1.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/49.5 kB ? eta -:--:--
   --------------------------------- ------ 41.0/49.5 kB 1.9 MB/s eta 0:00:01
   --------------------------------- ------ 41.0/49.5 kB 1.9 MB/s eta 0:00:01
   --------------------------------- ------ 41.0/49.5 kB 1.9 MB/s eta 0:00:01
   --------------------------------- ------ 41.0/49.5 kB 1.9 MB/s eta 0:00:01
   --------------------------------- ------ 41.0/49.5 kB 1.9 MB/s eta 0:00:01
   ---------------------------------------- 49.5/49.5 kB 157.1 kB/s eta 0:00:00
Using cached wcmatch-10.1-py3-none-any.whl (39 kB)
   ---------------------------------------- 0.0/388.2 kB ? eta 


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
import os
from typing import Literal
from tavily import TavilyClient
from deepagents import create_deep_agent
from langchain_openai import ChatOpenAI


# Load env
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

# Initialize OpenAI model
llm = ChatOpenAI(
    model="gpt-4o-mini",  # or any supported model
    api_key=os.environ["OPENAI_API_KEY"],
)

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Search the internet using Tavily"""
    return tavily_client.search(
        query=query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )



In [35]:
# System prompt to steer the agent to be an expert researcher
research_instructions = """You are an expert researcher. Your job is to conduct thorough research and then write a polished report.

You have access to an internet search tool as your primary means of gathering information.

## `internet_search`

Use this to run an internet search for a given query. You can specify the max number of results to return, the topic, and whether raw content should be included.
"""

agent = create_deep_agent(
    tools=[internet_search],
    system_prompt=research_instructions,
)

In [37]:
result = agent.invoke({"messages": [{"role": "user", "content": "What is langgraph?"}]})

# Print the agent's response
print(result["messages"][-1].content)

TypeError: "Could not resolve authentication method. Expected either api_key or auth_token to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"

In [1]:
import os
import sqlite3
import logging
from fastapi import FastAPI
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.sqlite import SqliteSaver

app = FastAPI()
agent = None

logging.basicConfig(level=logging.INFO)

@app.on_event("startup")
async def startup_event():
    global agent

    # --- Paths ---
    CURRENT_PATH = os.getcwd()                   # Repository root (auto)
    SQLITE_DB = "checkpoints.db"                # Persistent memory DB

    # --- Memory (persistent SQLite checkpoints) ---
    conn = sqlite3.connect(SQLITE_DB, check_same_thread=False)
    memory = SqliteSaver(conn)

    # --- LLM ---
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        api_key=os.environ["OPENAI_API_KEY"]
    )

    # --- Create Deep Agent ---
    agent = create_deep_agent(
        model=llm,
        system_prompt=(
            "You are a legacy code repository analyzer. "
            "Your task is to:\n"
            "1. Scan the entire repository recursively.\n"
            "2. Identify all code files and read them.\n"
            "3. Extract technologies used, dependencies, modules, and weak areas.\n"
            "4. Generate a complete TODO modernization plan.\n"
            "5. Save the plan as a Markdown file named `modernization_plan.md` "
            "in the repository root.\n"
            "Always produce a structured and professional modernization plan."
        ),
        backend=FilesystemBackend(root_dir=CURRENT_PATH),   # File access
        memory=memory,                                      # Persistent memory
        autonomous=True                                     # Auto planning + tool calls
    )

    logging.info("DeepAgent initialized successfully.")



ModuleNotFoundError: No module named 'langgraph.checkpoint.sqlite'

In [46]:
%pip install pythonnet


  Using cached pycparser-2.23-py3-none-any.whl.metadata (993 bytes)
   ---------------------------------------- 0.0/297.5 kB ? eta -:--:--
   ---------------------------------------- 0.0/297.5 kB ? eta -:--:--
   - -------------------------------------- 10.2/297.5 kB ? eta -:--:--
   - -------------------------------------- 10.2/297.5 kB ? eta -:--:--
   ----- --------------------------------- 41.0/297.5 kB 326.8 kB/s eta 0:00:01
   -------------------------- ------------- 194.6/297.5 kB 1.2 MB/s eta 0:00:01
   ---------------------------------------- 297.5/297.5 kB 1.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/56.4 kB ? eta -:--:--
   ---------------------------------------- 56.4/56.4 kB 2.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/183.6 kB ? eta -:--:--
   ----------------- ---------------------- 81.9/183.6 kB 4.5 MB/s eta 0:00:01
   ---------------------------------------- 183.6/183.6 kB 2.8 MB/s eta 0:00:00
Using cached pycparser-2.23-


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [48]:
import clr
import os

# Use current working directory
script_dir = os.getcwd()

# Add Roslyn DLLs
clr.AddReference(os.path.join(script_dir, "Microsoft.CodeAnalysis.CSharp.dll"))
clr.AddReference(os.path.join(script_dir, "Microsoft.CodeAnalysis.dll"))

from Microsoft.CodeAnalysis.CSharp import CSharpSyntaxTree
from Microsoft.CodeAnalysis import SyntaxKind

# Sample C# code
csharp_code = """
/// <summary>
/// Adds two integers
/// </summary>
int Add(int a, int b) {
    return a + b;
}
"""

# Parse C# code
tree = CSharpSyntaxTree.ParseText(csharp_code)
root = tree.GetRoot()

# Traverse methods
for node in root.DescendantNodes():
    if node.IsKind(SyntaxKind.MethodDeclaration):
        print("Method name:", node.Identifier.ValueText)
        print("Return type:", node.ReturnType.ToString())
        print("Parameters:")
        for param in node.ParameterList.Parameters:
            print(f"  {param.Identifier.ValueText} : {param.Type.ToString()}")
        # Print XML documentation comments
        trivia = node.GetLeadingTrivia()
        for t in trivia:
            if t.IsKind(SyntaxKind.SingleLineDocumentationCommentTrivia):
                print("Doc comment:", t.ToString())


FileNotFoundException: Unable to find assembly 'c:\Users\sam\Documents\projects\rag-project\langgraph-rag\src\poc1\backend\Microsoft.CodeAnalysis.CSharp.dll'.
   at Python.Runtime.CLRModule.AddReference(String name)

Function name: add
Return type: UNKNOWN
Parameters: [('a', 'int'), ('b', 'int')]


: 

In [11]:
import os
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain_openai import ChatOpenAI

CURRENT_PATH = os.getcwd()   # 👈 automatically picks the folder where the app is running
SQLITE_DB = "./memory_store.sqlite"

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=api_key
)

agent = create_deep_agent(
    model=llm,
    system_prompt=(
        "You are a legacy code repository analyzer. "
        "Scan the entire folder recursively, list all code files, read them, "
        "extract technologies used, dependencies, modules, and weaknesses. "
        "Finally, generate a TODO modernization plan in Markdown with clear steps."
    ),
    backend=FilesystemBackend(root_dir=CURRENT_PATH),   # 👈 uses current folder
    memory=SQLiteMemoryBackend(db_path=SQLITE_DB),      # 👈 persistent memory
    autonomous=True
)


NameError: name 'load_dotenv' is not defined

In [3]:
%pip install langgraph-checkpoint

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
%pip install langgraph-checkpoint-sqlite

  Using cached langgraph_checkpoint_sqlite-3.0.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached aiosqlite-0.21.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached sqlite_vec-0.1.6-py3-none-win_amd64.whl.metadata (198 bytes)
Using cached langgraph_checkpoint_sqlite-3.0.0-py3-none-any.whl (32 kB)
Using cached aiosqlite-0.21.0-py3-none-any.whl (15 kB)
Using cached sqlite_vec-0.1.6-py3-none-win_amd64.whl (281 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import logging
from dotenv import load_dotenv

from deepagents import create_deep_agent
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langgraph.store.memory import InMemoryStore
from langchain_openai import ChatOpenAI

logging.basicConfig(level=logging.INFO)

# Load OpenAI API key
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

# Paths
CURRENT_PATH = os.getcwd()                       # Scan current working directory
MEMORY_DIR = os.path.join(CURRENT_PATH, "memories")
os.makedirs(MEMORY_DIR, exist_ok=True)          # Persistent memory directory

# Persisten tmemory store
store = InMemoryStore()

# CompositeBackend: ephemeral workspace + persistent memory
def backend(rt):
    return CompositeBackend(
        default=StateBackend(rt),                  # scratch workspace
        routes={"/memories/": StoreBackend(rt)}    # persistent memory for documentation & TODOs
    )

# LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=api_key
)

# ---- Create Deep Agent (TodoListMiddleware included by default) ----
agent = create_deep_agent(
    model=llm,
    system_prompt=(
        f"You are a legacy code repository analyzer.\n"
        f"Scan the repository at {CURRENT_PATH} recursively.\n"
        "Detect all code files, modules, dependencies, problems, and risk areas.\n"
        f"Save module documentation in /memories/ as individual Markdown files.\n"
        f"Maintain /memories/modernization_plan.md and /memories/todo_plan.md incrementally.\n"
        "Use the built-in write_todos tool to manage the TODO list automatically."
    ),
    backend=backend,
    store=store
)

print("✅ DeepAgent ready: Legacy code documentation + TODO planning + persistent memory")

# --- Optional: Run agent to generate docs and TODOs ---
response = agent.invoke({"input": "Begin legacy code documentation and generate TODOs"})
print(response)


📂 Session workspace: c:\Users\sam\Documents\projects\rag-project\langgraph-rag\src\poc1\backend\workspaces\user_1234
📂 Memory folder: c:\Users\sam\Documents\projects\rag-project\langgraph-rag\src\poc1\backend\workspaces\user_1234\memories


✅ Copied source folder to session workspace: c:\Users\sam\Documents\projects\rag-project\langgraph-rag\src\poc1\backend\workspaces\user_1234\repo
✅ Agent ready for session: user_1234


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


{'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 4441, 'total_tokens': 4462, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 4352}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b547601dbd', 'id': 'chatcmpl-Cj82Z2ujiyQ479ivi496MmeHedbaz', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--c02d5e9e-860a-45f8-9e8a-27f810bcbf8d-0', tool_calls=[{'name': 'glob', 'args': {'pattern': '**/*', 'path': '/workspace/repo'}, 'id': 'call_3DnYd0fDmyc9pNhyIHisclj4', 'type': 'tool_call'}], usage_metadata={'input_tokens': 4441, 'output_tokens': 21, 'total_tokens': 4462, 'input_token_details': {'audio': 0, 'cache_read': 4352}, 'output_token_details': {'audio': 0, 'reaso

In [36]:
import os
import logging
from pathlib import Path
from dotenv import load_dotenv

from deepagents import create_deep_agent
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from deepagents.middleware.filesystem import FilesystemMiddleware
from langgraph.store.memory import InMemoryStore
from langchain_openai import ChatOpenAI

logging.basicConfig(level=logging.INFO)

# --- Load OpenAI API key ---
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

# --- Paths / session workspace ---
CURRENT_PATH = Path.cwd()
MEMORY_DIR = CURRENT_PATH / "memories"
MEMORY_DIR.mkdir(exist_ok=True)

print(f"📂 Current folder to scan: {CURRENT_PATH}")
print(f"📂 Memory folder: {MEMORY_DIR}")

# --- Persistent memory store ---
store = InMemoryStore()

# --- CompositeBackend: scratch workspace + persistent memory ---
def backend(rt):
    return CompositeBackend(
        default=StateBackend(rt),                  # ephemeral scratch workspace
        routes={"/memories/": StoreBackend(rt)}    # persistent memory for docs & TODOs
    )

# --- LLM ---
llm = ChatOpenAI(model="gpt-4o-mini", api_key=api_key)

# --- Virtual path mapping (for agent filesystem tools) ---
VIRTUAL_PATH = "/workspace/repo"

# --- Create DeepAgent with FilesystemMiddleware ---
agent = create_deep_agent(
    model=llm,
    system_prompt=(
        f"You are a legacy code repository analyzer.\n"
        f"Scan the repository at {VIRTUAL_PATH} recursively.\n"
        "Detect all code files, modules, dependencies, problems, and risk areas.\n"
        f"Save module documentation in /memories/ as individual Markdown files.\n"
        f"Maintain /memories/modernization_plan.md and /memories/todo_plan.md incrementally.\n"
        "Use the built-in write_todos tool to manage the TODO list automatically."
    ),
    backend=backend,
    store=store,
    middleware=[
        FilesystemMiddleware(
            system_prompt="Read code files, analyze them, and generate explanations and TODOs in /memories/.",
            custom_tool_descriptions={
                "ls": "List files and folders to find code files",
                "read_file": "Read code files and explain their functionality",
                "write_file": "Write Markdown summaries or TODOs in /memories/"
            }
        )
    ]
)

print("✅ DeepAgent ready: Legacy code documentation + TODO planning + persistent memory")

# --- Optional: Run agent to scan current folder and generate docs + TODOs ---
response = agent.invoke({
    "input": f"Scan all files under {VIRTUAL_PATH}, generate a detailed explanation for each code file, and write the documentation and TODO list to /memories/"
})

print("Agent response:")
print(response)

# --- List generated Markdown files in /memories/ ---
md_files = list(MEMORY_DIR.glob("*.md"))
print(f"Generated {len(md_files)} markdown files in /memories/:")
for f in md_files:
    print(f" - {f}")


📂 Current folder to scan: c:\Users\sam\Documents\projects\rag-project\langgraph-rag\src\poc1\backend
📂 Memory folder: c:\Users\sam\Documents\projects\rag-project\langgraph-rag\src\poc1\backend\memories


AssertionError: Please remove duplicate middleware instances.